In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import label_binarize
import joblib
import os

# Ensure model folder exists
os.makedirs("/home/cloud/Desktop/model", exist_ok=True)

# Load dataset
df = pd.read_csv("/home/cloud/Desktop/winequality-red.csv", sep=";")

X = df.drop("quality", axis=1)
y = df["quality"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

results = {}
classes = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=classes)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # ✅ Save trained model as .pkl
    filename = f"/home/cloud/Desktop/model/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(model, filename)
    print(f"Saved: {filename}")   # Debug message to confirm saving

    # Probabilities for AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        auc = roc_auc_score(y_test_bin, y_prob, multi_class="ovr")
    else:
        auc = "N/A"

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": auc,
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "F1": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

# Save results table
results_df = pd.DataFrame(results).T
print(results_df)
results_df.to_csv("/home/cloud/Desktop/test_data.csv", index=True)

Saved: /home/cloud/Desktop/model/logistic_regression.pkl
Saved: /home/cloud/Desktop/model/decision_tree.pkl
Saved: /home/cloud/Desktop/model/knn.pkl
Saved: /home/cloud/Desktop/model/naive_bayes.pkl


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Saved: /home/cloud/Desktop/model/random_forest.pkl
                     Accuracy       AUC  Precision    Recall        F1  \
Logistic Regression  0.575000  0.846110   0.561804  0.575000  0.551084   
Decision Tree        0.578125  0.616029   0.574208  0.578125  0.575791   
KNN                  0.546875  0.674612   0.522388  0.546875  0.530905   
Naive Bayes          0.546875  0.816657   0.542588  0.546875  0.543498   
Random Forest        0.656250  0.779961   0.625418  0.656250  0.638198   

                          MCC  
Logistic Regression  0.309047  
Decision Tree        0.341925  
KNN                  0.269449  
Naive Bayes          0.301989  
Random Forest        0.447753  


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
